# 04 - Evaluation dan Profiling Segmen
Evaluasi terhadap target SMART, stabilitas antar seed, profil segmen, dan rekomendasi promosi.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rfm_kmeans import config
cfg = config.load_config()
SEED = cfg["seed"]
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)
from rfm_kmeans import modeling, evaluation, profiling, visualization
OUT = config.resolve(cfg, "processed_dir")
TAB = config.resolve(cfg, "tables_dir"); FIG = config.resolve(cfg, "figures_dir")
rfm = pd.read_csv(OUT / "rfm.csv")
X = pd.read_csv(OUT / "rfm_scaled.csv", index_col=0)
k_scan = pd.read_csv(TAB / "k_selection.csv")
k = modeling.choose_k(k_scan, cfg["modeling"]["k_candidates"])
model = modeling.fit_kmeans(X, k, SEED, cfg["modeling"]["n_init"])
labels = model.labels_

## Metrik dan target SMART

In [2]:
checks = evaluation.check_smart_targets(X, labels, cfg["evaluation"])
checks

,Target,Nilai,Tercapai
0,Silhouette >= 0.40,0.32,False
1,Davies-Bouldin < 1.00,1.01,False
2,Jumlah cluster 3-6,5.00,True
3,Cluster terkecil >= 5%,10.46,True


## Stabilitas (Adjusted Rand Index antar seed)

In [3]:
stab = evaluation.stability_ari(X, k, cfg["evaluation"]["stability_seeds"])
stab

{'ari_mean': 0.9924028769408118, 'ari_min': 0.9873381282346863}

## Profil segmen

In [4]:
prof = profiling.profile_segments(rfm, labels)
mapping = profiling.suggest_label(prof)   # usulan otomatis, WAJIB dicek manual
prof["Segmen"] = prof.index.map(mapping)
prof

,Pelanggan,Recency_mean,Frequency_mean,Monetary_mean,Monetary_total,Pelanggan_%,Pendapatan_%,Segmen
Cluster,,,,,,,,
0,62,21.85,16.06,"9,630.80","597,109.41",10.46,54.68,Champions
1,124,13.29,5.25,"1,721.37","213,449.30",20.91,19.55,Loyal
2,128,39.95,1.34,277.14,"35,473.44",21.59,3.25,Potential / New
3,125,247.72,1.22,275.10,"34,386.94",21.08,3.15,Lost
4,154,91.51,3.29,"1,373.39","211,501.61",25.97,19.37,Lost


In [5]:
visualization.plot_segment_summary(prof, FIG / "segment_summary.png")
plt.show()

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_96872/1366174164.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Rekomendasi strategi promosi

In [6]:
result = profiling.add_segment_names(rfm, labels, mapping)
result.groupby(["Segmen", "Strategi"]).size().rename("Pelanggan").reset_index()

,Segmen,Strategi,Pelanggan
0,Champions,"Program loyalitas, akses awal produk baru",62
1,Lost,"Reaktivasi berbiaya rendah, misalnya email",279
2,Loyal,"Upselling, cross-selling, dan bundling",124
3,Potential / New,"Voucher selamat datang, edukasi produk",128


In [7]:
result.to_csv(OUT / "rfm_clustered.csv", index=False)
prof.to_csv(TAB / "segment_profile.csv")

## Kesimpulan
Isi setelah dijalankan: apakah target Silhouette dan Davies-Bouldin tercapai, seberapa stabil cluster, segmen mana yang paling berkontribusi pada pendapatan, dan strategi untuk tiap segmen. Jika target belum tercapai, tuliskan sebagai temuan dan diskusikan penyebabnya (outlier, pembeli grosir, bentuk cluster tidak bulat).